In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import joblib
import tensorflow as tf
import pickle
import os

In [3]:
# Load data
df = pd.read_csv("D:\Sunfibo internship\wearble health monitoring\wearable_health_monitoring_ai\health.cs", parse_dates=['Timestamp'])

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19676\911752246.py:2: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("D:\Sunfibo internship\wearble health monitoring\wearable_health_monitoring_ai\health.cs", parse_dates=['Timestamp'])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19676\911752246.py:2: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("D:\Sunfibo internship\wearble health monitoring\wearable_health_monitoring_ai\health.cs", parse_dates=['Timestamp'])


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Sunfibo internship\\wearble health monitoring\\wearable_health_monitoring_ai\\health.cs'

In [5]:
# Missing values fill karna, e.g., forward fill
df['Heart Rate (BPM)'] = df['Heart Rate (BPM)'].fillna(method='ffill')
df['SpO2 (%)'] = df['SpO2 (%)'].fillna(method='ffill')


C:\Users\MI\AppData\Local\Temp\ipykernel_21228\2506706797.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Heart Rate (BPM)'] = df['Heart Rate (BPM)'].fillna(method='ffill')
C:\Users\MI\AppData\Local\Temp\ipykernel_21228\2506706797.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['SpO2 (%)'] = df['SpO2 (%)'].fillna(method='ffill')


In [6]:
# Define function for labeling
def label_row(row):
    hr = row['Heart Rate (BPM)']
    spo2 = row['SpO2 (%)']
    
    if (hr < 60 or hr > 100) or (spo2 < 95):
        return 'Abnormal'
    else:
        return 'Normal'

In [7]:
# Apply labeling
df['Label'] = df.apply(label_row, axis=1)

print(df[['Timestamp', 'Heart Rate (BPM)', 'SpO2 (%)', 'Label']].head())


            Timestamp  Heart Rate (BPM)  SpO2 (%)     Label
0 2025-04-28 07:55:00               NaN       NaN    Normal
1 2025-04-28 08:00:00               NaN       NaN    Normal
2 2025-04-28 08:05:00               NaN       NaN    Normal
3 2025-04-28 08:10:00               NaN       NaN    Normal
4 2025-04-28 08:15:00              89.0      93.0  Abnormal


In [8]:
# Features and labels
X = df[['Steps', 'Heart Rate (BPM)', 'SpO2 (%)']]
y = df['Label']

In [9]:
# Create classification target (0 = normal, 1 = abnormal)
df['Status'] = np.where((df['Heart Rate (BPM)'] < 60) | 
                         (df['Heart Rate (BPM)'] > 100), 1, 0)

In [10]:
# Encode label
y = y.map({'Normal':0, 'Abnormal':1})

In [11]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
# Model
model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [13]:

# Prediction
y_pred = model.predict(X_test)

In [14]:

# Evaluation
print(classification_report(y_test, y_pred))
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       145
           1       1.00      1.00      1.00        63

    accuracy                           1.00       208
   macro avg       1.00      1.00      1.00       208
weighted avg       1.00      1.00      1.00       208

Accuracy: 1.0


In [15]:
print(df['Label'].value_counts())



Label
Normal      737
Abnormal    300
Name: count, dtype: int64


In [16]:
import pickle

# Save the model to a .pkl file
with open('random_forest_model.pkl', 'wb') as file:
    pickle.dump(model, file)